# 🌳 Étape 4 — Bootstrap CI pour les métriques principales
## Renforcement méthodologique style MDPI Agriculture

**À exécuter APRÈS** le notebook unifié Étape 1+2 (les variables `df`, `FEATURES_ALL`, `FEATURES_BASE`, `FEATURES_TEMPORAL`, `make_models()`, `groups`, `y_3` doivent être en mémoire).

**Objectif :** ajouter des intervalles de confiance bootstrap à 95% pour les 3 configurations principales, comme le fait Aguirre-Munizaga et al. (2026) dans *Agriculture* MDPI. Cela renforce la rigueur statistique du papier.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 — Bootstrap 95% CI pour Macro F1
# ═══════════════════════════════════════════════════════════════
#
# Protocole :
# 1. Pour chaque configuration (A, B, C), prédire sur le test set GroupKFold
# 2. Concaténer y_true et y_pred sur tous les folds
# 3. Bootstrap : 2000 resamples avec remise de paires (y_true, y_pred)
# 4. Reporter le CI à 95% (percentiles 2.5 et 97.5)
# ═══════════════════════════════════════════════════════════════

import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RNG_SEED = 42
N_BOOTSTRAP = 2000

def bootstrap_ci_f1(y_true, y_pred, n_bootstrap=2000, seed=42):
    """
    Calcule l'intervalle de confiance bootstrap à 95% pour Macro F1, Accuracy, Balanced Accuracy.
    Resampling avec remise de paires (y_true, y_pred).
    """
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    f1_scores  = []
    acc_scores = []
    bal_scores = []

    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        y_t = y_true[idx]
        y_p = y_pred[idx]
        # Vérifier qu'on a au moins 2 classes dans l'échantillon
        if len(np.unique(y_t)) < 2:
            continue
        f1_scores.append(f1_score(y_t, y_p, average='macro', zero_division=0))
        acc_scores.append(accuracy_score(y_t, y_p))
        bal_scores.append(balanced_accuracy_score(y_t, y_p))

    return {
        'f1_mean'  : np.mean(f1_scores),
        'f1_ci'    : (np.percentile(f1_scores, 2.5), np.percentile(f1_scores, 97.5)),
        'acc_mean' : np.mean(acc_scores),
        'acc_ci'   : (np.percentile(acc_scores, 2.5), np.percentile(acc_scores, 97.5)),
        'bal_mean' : np.mean(bal_scores),
        'bal_ci'   : (np.percentile(bal_scores, 2.5), np.percentile(bal_scores, 97.5)),
    }

def collect_predictions(X, y, groups, model_name='XGBoost'):
    """Collecte les prédictions sur tous les folds GroupKFold(5)."""
    gkf = GroupKFold(n_splits=5)
    y_true_all, y_pred_all = [], []

    for train_idx, test_idx in gkf.split(X, y, groups):
        pipe = make_models()[model_name]
        pipe.fit(X[train_idx], y[train_idx])
        y_pred = pipe.predict(X[test_idx])
        y_true_all.extend(y[test_idx])
        y_pred_all.extend(y_pred)

    return np.array(y_true_all), np.array(y_pred_all)

print(f'Calcul des bootstrap CI ({N_BOOTSTRAP} resamples par config)')
print('Cela prend environ 1-2 minutes...\n')

# ── Configuration A : Static 4 classes ──
y_4 = df['label'].values  # ou y si déjà défini
X_a = df[FEATURES_BASE].values
y_true_a, y_pred_a = collect_predictions(X_a, y_4, groups)
ci_a = bootstrap_ci_f1(y_true_a, y_pred_a, N_BOOTSTRAP)
print(f'CONFIG A — Static, 4 classes :')
print(f'  Macro F1 = {ci_a["f1_mean"]:.3f} [{ci_a["f1_ci"][0]:.3f}, {ci_a["f1_ci"][1]:.3f}]')
print(f'  Accuracy = {ci_a["acc_mean"]:.3f} [{ci_a["acc_ci"][0]:.3f}, {ci_a["acc_ci"][1]:.3f}]')
print(f'  Bal Acc  = {ci_a["bal_mean"]:.3f} [{ci_a["bal_ci"][0]:.3f}, {ci_a["bal_ci"][1]:.3f}]')
print()

# ── Configuration B : Enriched 4 classes ──
X_b = df[FEATURES_ALL].values
y_true_b, y_pred_b = collect_predictions(X_b, y_4, groups)
ci_b = bootstrap_ci_f1(y_true_b, y_pred_b, N_BOOTSTRAP)
print(f'CONFIG B — Enriched, 4 classes :')
print(f'  Macro F1 = {ci_b["f1_mean"]:.3f} [{ci_b["f1_ci"][0]:.3f}, {ci_b["f1_ci"][1]:.3f}]')
print(f'  Accuracy = {ci_b["acc_mean"]:.3f} [{ci_b["acc_ci"][0]:.3f}, {ci_b["acc_ci"][1]:.3f}]')
print(f'  Bal Acc  = {ci_b["bal_mean"]:.3f} [{ci_b["bal_ci"][0]:.3f}, {ci_b["bal_ci"][1]:.3f}]')
print()

# ── Configuration C : Enriched 3 classes ──
X_c = df[FEATURES_ALL].values
y_true_c, y_pred_c = collect_predictions(X_c, y_3, groups)
ci_c = bootstrap_ci_f1(y_true_c, y_pred_c, N_BOOTSTRAP)
print(f'CONFIG C — Enriched, 3 classes :')
print(f'  Macro F1 = {ci_c["f1_mean"]:.3f} [{ci_c["f1_ci"][0]:.3f}, {ci_c["f1_ci"][1]:.3f}]')
print(f'  Accuracy = {ci_c["acc_mean"]:.3f} [{ci_c["acc_ci"][0]:.3f}, {ci_c["acc_ci"][1]:.3f}]')
print(f'  Bal Acc  = {ci_c["bal_mean"]:.3f} [{ci_c["bal_ci"][0]:.3f}, {ci_c["bal_ci"][1]:.3f}]')
print()

print('✅ Cellule 1 — Bootstrap CI calculés')
print()
print('=' * 65)
print('  TABLE 1 RÉVISÉE — À copier dans le manuscrit')
print('=' * 65)
print()
print(f'| Configuration | N feat | N cls | Macro F1 [95% CI]                    |')
print(f'|---|---|---|---|')
print(f'| A — Static    | 8  | 4 | {ci_a["f1_mean"]:.3f} [{ci_a["f1_ci"][0]:.3f}, {ci_a["f1_ci"][1]:.3f}] |')
print(f'| B — Enriched  | 24 | 4 | {ci_b["f1_mean"]:.3f} [{ci_b["f1_ci"][0]:.3f}, {ci_b["f1_ci"][1]:.3f}] |')
print(f'| C — Enriched  | 24 | 3 | {ci_c["f1_mean"]:.3f} [{ci_c["f1_ci"][0]:.3f}, {ci_c["f1_ci"][1]:.3f}] |')